<a href="https://colab.research.google.com/github/KimJong-sub/shrek/blob/main/weather_brief_update.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
#1. 라이브러리 설치 및 import ==========
!pip install requests beautifulsoup4 lxml ipywidgets --quiet

import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from collections import defaultdict, Counter
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

#2. 설정값 ==========
API_KEY      = "e7eba533bc9a7b9487df4b77093e4332c42aa5bf0eeb316dac315ab39a1b7315"
FORECAST_URL = "https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst"
REALTIME_URL = "https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst"

REGION_DATA = {
    "서울특별시":     { "서울":   (60, 127, ["서울", "경기"]) },
    "부산광역시":     { "부산":   (98, 76,  ["부산", "경남"]) },
    "대구광역시":     { "대구":   (89, 90,  ["대구", "경북"]) },
    "인천광역시":     { "인천":   (55, 124, ["인천", "경기"]) },
    "광주광역시":     { "광주":   (58, 74,  ["광주", "전남"]) },
    "대전광역시":     { "대전":   (67, 100, ["대전", "충청", "세종"]) },
    "울산광역시":     { "울산":   (102, 84, ["울산", "경남"]) },
    "세종특별자치시": { "세종":   (66, 103, ["세종", "충청"]) },
    "경기도": {
        "수원": (60, 121, ["경기"]), "성남": (62, 123, ["경기"]),
        "고양": (57, 128, ["경기"]), "용인": (62, 120, ["경기"]),
        "안양": (59, 123, ["경기"]), "안산": (57, 121, ["경기"]),
        "평택": (62, 114, ["경기"]), "의정부": (61, 130, ["경기"]),
    },
    "강원도": {
        "춘천": (73, 134, ["강원"]), "원주": (76, 122, ["강원"]),
        "강릉": (92, 131, ["강원"]), "속초": (87, 141, ["강원"]),
    },
    "충청북도": {
        "청주": (69, 106, ["충청", "충북"]), "충주": (76, 114, ["충청", "충북"]),
        "제천": (81, 119, ["충청", "충북"]),
    },
    "충청남도": {
        "천안": (63, 110, ["충청", "충남"]), "공주": (63, 102, ["충청", "충남"]),
        "홍성": (55, 103, ["충청", "충남"]), "아산": (60, 110, ["충청", "충남"]),
    },
    "전라북도": {
        "전주": (63, 89,  ["전북", "전라"]), "군산": (56, 92,  ["전북", "전라"]),
        "익산": (60, 91,  ["전북", "전라"]), "남원": (68, 80,  ["전북", "전라"]),
    },
    "전라남도": {
        "목포": (50, 67,  ["전남", "전라"]), "여수": (73, 66,  ["전남", "전라"]),
        "순천": (70, 70,  ["전남", "전라"]), "나주": (56, 71,  ["전남", "전라"]),
    },
    "경상북도": {
        "포항": (102, 94, ["경북", "경상"]), "경주": (100, 91, ["경북", "경상"]),
        "구미": (84,  96, ["경북", "경상"]), "안동": (91, 106, ["경북", "경상"]),
    },
    "경상남도": {
        "창원": (91, 77, ["경남", "경상"]), "진주": (81, 75, ["경남", "경상"]),
        "통영": (87, 68, ["경남", "경상"]), "김해": (95, 77, ["경남", "경상"]),
    },
    "제주특별자치도": {
        "제주": (52, 38, ["제주"]), "서귀포": (52, 33, ["제주"]),
    },
}

print("✅ 설정 완료")

#3. 공통 함수 정의 ==========
def get_base_time():
    now = datetime.now()
    base_times = [2, 5, 8, 11, 14, 17, 20, 23]
    hour = now.hour
    adjusted_hour = hour if now.minute >= 10 else hour - 1
    base_hour = max([t for t in base_times if t <= adjusted_hour], default=23)
    if adjusted_hour < 2:
        base_hour = 23
        now = now - timedelta(days=1)
    return now.strftime("%Y%m%d"), f"{base_hour:02d}00"

def call_api(url, params):
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        if data["response"]["header"]["resultCode"] == "00":
            return data["response"]["body"]["items"]["item"]
        else:
            print(f"[API 오류] {data['response']['header']['resultMsg']}")
            return None
    except Exception as e:
        print(f"[연결 오류] {e}")
        return None

def sky_label(v):
    return {"1": "맑음", "3": "구름많음", "4": "흐림"}.get(str(v), "-")

def pty_label(v):
    return {"0": "없음", "1": "비", "2": "비/눈", "3": "눈", "4": "소나기"}.get(str(v), "-")

def wind_dir_label(deg):
    dirs = ["북","북북동","북동","동북동","동","동남동","남동","남남동",
            "남","남남서","남서","서남서","서","서북서","북서","북북서"]
    try:
        return dirs[round(float(deg) / 22.5) % 16] + "풍"
    except:
        return ""

def fire_risk(temp, humidity, wind):
    try:
        t, h, w = float(temp), float(humidity), float(wind)
        score = 0
        if t > 25:    score += 2
        elif t > 15:  score += 1
        if h < 30:    score += 3
        elif h < 50:  score += 2
        elif h < 65:  score += 1
        if w > 9:     score += 3
        elif w > 5:   score += 2
        elif w > 3:   score += 1
        if score >= 6: return "매우높음", "화기 취급 전면 금지, 소방시설 즉시 점검"
        if score >= 4: return "높음",     "야외 화기 사용 자제, 소화기 비치 확인"
        if score >= 2: return "보통",     "일반적 화기 주의 사항 준수"
        return "낮음", "이상 없음"
    except:
        return "산출불가", "-"

def visibility_from_weather(sky, pty, wind):
    try:
        w = float(wind)
    except:
        w = 0
    if str(pty) in ["1", "4"]:          return "🌧️ 약 1~3 km (우천)"
    if str(pty) in ["2", "3"]:          return "🌨️ 약 0.5~2 km (눈)"
    if str(sky) == "4" and w > 5:       return "🌫️ 약 3~5 km (흐림+강풍)"
    if str(sky) == "4":                 return "☁️ 약 5~8 km (흐림)"
    if str(sky) == "3":                 return "⛅ 약 8~15 km (구름많음)"
    return "☀️ 10 km 이상 (양호)"

def accident_measures(pty, wind, temp):
    measures = []
    try:
        w = float(wind)
        t = float(temp)
    except:
        w, t = 0, 15
    if str(pty) in ["1", "4"]:
        measures += ["우천 시 도로 결빙 및 낙상 위험 증가 → 차량 서행 및 도보 이동 시 주의",
                     "우의 및 방수장비 착용, 야외 훈련 일정 재검토"]
    if str(pty) in ["2", "3"]:
        measures += ["강설 시 제설 작업 즉시 실시, 낙상·교통사고 주의",
                     "동파 위험 시설(수도관·차량) 사전 점검"]
    if w >= 9:
        measures += ["강풍 주의보 수준 → 고소작업 및 천막·구조물 고정 여부 점검",
                     "낙하물 위험 구역 접근 통제"]
    elif w >= 5:
        measures += ["바람 강함 → 경량 장비 고정 및 외부 훈련 시 방풍 조치"]
    if t <= 0:
        measures += ["영하 기온 → 동상·저체온증 예방, 내한 장비 착용 필수",
                     "차량 시동 예열 및 결빙 노면 확인"]
    elif t <= 5:
        measures += ["저온 주의 → 방한복 착용 권고, 외부 작업 시 체온 유지"]
    if t >= 33:
        measures += ["폭염 주의보 수준 → 충분한 수분 보충, 야외 훈련 시간 조정",
                     "온열질환 의심 증상 발생 시 즉시 보고 조치"]
    elif t >= 28:
        measures += ["고온 주의 → 야외 작업 전 열 스트레스 점검, 휴식 시간 확보"]
    if not measures:
        measures = ["현재 특이 기상 위험 없음 — 일반 안전 수칙 준수"]
    return measures

def get_weather_warning_rss(keywords):
    warn_url = "https://www.weather.go.kr/w/rss/warning.do"
    try:
        resp = requests.get(warn_url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        resp.encoding = "utf-8"
        soup = BeautifulSoup(resp.text, "lxml-xml")
        items = []
        for item in soup.find_all("item"):
            title    = item.find("title")
            desc     = item.find("description")
            pub_date = item.find("pubDate")
            t  = title.get_text(strip=True)    if title    else ""
            d  = desc.get_text(strip=True)     if desc     else ""
            dt = pub_date.get_text(strip=True) if pub_date else ""
            if any(kw in t + d for kw in keywords):
                items.append({"title": t, "desc": d, "date": dt})
        return items
    except Exception as e:
        return None

print("✅ 함수 정의 완료")

#4. 브리핑 렌더링 함수 ==========
def fetch_and_render(region_name, nx, ny, warn_keywords):
    """
    ✅ 핵심 수정:
    - datetime.now()를 함수 진입 시점에 새로 호출하여 항상 최신 시각 사용
    - API 파라미터(base_date, base_time, base_time for realtime)도 이 시점 기준으로 산출
    """
    today     = datetime.now()           # ← 버튼 클릭 시점의 현재 시각
    today_str = today.strftime("%Y%m%d")
    now_hour  = today.strftime("%H") + "00"
    base_date, base_time = get_base_time()

    # ── API 호출 ────────────────────────────────────────────────────
    fcst_items = call_api(FORECAST_URL, {
        "serviceKey": API_KEY, "numOfRows": 1000, "pageNo": 1,
        "dataType": "JSON", "base_date": base_date, "base_time": base_time,
        "nx": nx, "ny": ny,
    })
    now_items = call_api(REALTIME_URL, {
        "serviceKey": API_KEY, "numOfRows": 100, "pageNo": 1,
        "dataType": "JSON", "base_date": today_str, "base_time": now_hour,
        "nx": nx, "ny": ny,
    })
    warn_items = get_weather_warning_rss(warn_keywords)

    # ── 실황 파싱 ───────────────────────────────────────────────────
    now_data = {}
    if now_items:
        for it in now_items:
            now_data[it["category"]] = it["obsrValue"]
    cur_temp = now_data.get("T1H", "N/A")
    cur_hum  = now_data.get("REH", "N/A")
    cur_wind = now_data.get("WSD", "N/A")
    cur_wdir = now_data.get("VEC", "N/A")
    cur_pty  = now_data.get("PTY", "0")

    # ── 예보 파싱 ───────────────────────────────────────────────────
    fcst_by_dt = defaultdict(dict)
    if fcst_items:
        for it in fcst_items:
            key = f"{it['fcstDate']}_{it['fcstTime']}"
            fcst_by_dt[key][it["category"]] = it["fcstValue"]

    daily = defaultdict(lambda: {"TMX": None, "TMN": None, "SKY": [], "PTY": [], "POP": []})
    for key, vals in fcst_by_dt.items():
        d = key[:8]
        if "TMX" in vals: daily[d]["TMX"] = vals["TMX"]
        if "TMN" in vals: daily[d]["TMN"] = vals["TMN"]
        if "SKY" in vals: daily[d]["SKY"].append(vals["SKY"])
        if "PTY" in vals: daily[d]["PTY"].append(vals["PTY"])
        if "POP" in vals: daily[d]["POP"].append(int(vals["POP"]))

    def day_summary(ds):
        dd = daily.get(ds, {})
        sky = Counter(dd.get("SKY", [])).most_common(1)[0][0] if dd.get("SKY") else "N/A"
        pty = Counter(dd.get("PTY", [])).most_common(1)[0][0] if dd.get("PTY") else "N/A"
        pop = max(dd.get("POP", [])) if dd.get("POP") else "N/A"
        return sky, pty, pop, dd.get("TMX", "N/A"), dd.get("TMN", "N/A")

    today_sky, today_pty, today_pop, today_tmx, today_tmn = day_summary(today_str)
    tomorrow = (today + timedelta(days=1)).strftime("%Y%m%d")
    tmr_sky, tmr_pty, tmr_pop, tmr_tmx, tmr_tmn = day_summary(tomorrow)

    hour_temps = {}
    for key, vals in fcst_by_dt.items():
        if key[:8] == today_str and "TMP" in vals:
            hour_temps[key[9:]] = vals["TMP"]

    # ── 계산 ────────────────────────────────────────────────────────
    try:
        chill = f"{float(cur_temp) - float(cur_wind) * 0.7:.1f}°C"
    except:
        chill = "─"
    f_level, f_action = fire_risk(cur_temp, cur_hum, cur_wind)
    vis_str = visibility_from_weather(today_sky, cur_pty, cur_wind)
    m_list  = accident_measures(cur_pty, cur_wind, cur_temp)

    # ── HTML 빌더 헬퍼 ──────────────────────────────────────────────
    CARD = ("background:#fff;border-radius:12px;border:0.5px solid #ddd;"
            "padding:14px 16px;margin-bottom:10px;font-family:sans-serif;")
    SEC_NORMAL = ("font-size:14px;color:#1a3a5c;letter-spacing:.5px;"
                  "margin-bottom:10px;font-weight:700;")
    SEC_DANGER = ("display:inline-block;font-size:14px;font-weight:800;"
                  "letter-spacing:.5px;margin-bottom:12px;"
                  "background:#A32D2D;color:#fff;padding:4px 12px;border-radius:6px;")
    ROW = ("display:flex;justify-content:space-between;align-items:center;"
           "padding:5px 0;border-bottom:0.5px solid #eee;")
    LBL = "font-size:13px;color:#666;"
    VAL = "font-size:13px;font-weight:500;color:#111;text-align:right;"

    def card(title, body, danger=False):
        s = SEC_DANGER if danger else SEC_NORMAL
        return f'<div style="{CARD}"><div style="{s}">{title}</div>{body}</div>'

    def row(label, value, val_style=""):
        return (f'<div style="{ROW}">'
                f'<span style="{LBL}">{label}</span>'
                f'<span style="{VAL}{val_style}">{value}</span></div>')

    def stat2(l1, v1, s1, l2, v2, s2):
        def box(l, v, s):
            return (f'<div style="background:#f7f7f7;border-radius:8px;padding:10px 12px;flex:1;">'
                    f'<div style="font-size:11px;color:#888;margin-bottom:4px;">{l}</div>'
                    f'<div style="font-size:20px;font-weight:500;color:#111;">{v}</div>'
                    f'<div style="font-size:11px;color:#aaa;margin-top:2px;">{s}</div></div>')
        return (f'<div style="display:flex;gap:8px;margin-bottom:8px;">'
                f'{box(l1,v1,s1)}{box(l2,v2,s2)}</div>')

    def temp_bar_html(label, val, min_t=5, max_t=35):
        try:
            pct = max(0, min(100, int((float(val) - min_t) / (max_t - min_t) * 100)))
        except:
            pct = 0
        return (f'<div style="display:flex;align-items:center;gap:8px;margin:4px 0;">'
                f'<span style="font-size:12px;color:#888;width:32px;flex-shrink:0;">{label}</span>'
                f'<div style="flex:1;height:6px;background:#f0f0f0;border-radius:3px;overflow:hidden;">'
                f'<div style="width:{pct}%;height:100%;border-radius:3px;background:#378ADD;"></div></div>'
                f'<span style="font-size:12px;font-weight:500;width:38px;text-align:right;">{val}°C</span></div>')

    def fire_badge(level):
        color_map = {
            "매우높음": ("background:#FCEBEB;color:#A32D2D;", "매우높음"),
            "높음":     ("background:#FAECE7;color:#993C1D;", "높음"),
            "보통":     ("background:#FAEEDA;color:#854F0B;", "보통"),
            "낮음":     ("background:#EAF3DE;color:#3B6D11;", "낮음"),
        }
        st, txt = color_map.get(level, ("background:#eee;color:#333;", level))
        return (f'<span style="display:inline-block;padding:2px 10px;border-radius:20px;'
                f'font-size:12px;font-weight:500;{st}">{txt}</span>')

    def measure_items_html(m_list):
        rows = ""
        for i, m in enumerate(m_list, 1):
            rows += (f'<div style="display:flex;gap:8px;padding:8px;'
                     f'background:#fff8f8;border-radius:4px;margin-bottom:3px;align-items:flex-start;">'
                     f'<span style="font-size:12px;color:#A32D2D;font-weight:700;'
                     f'width:20px;flex-shrink:0;margin-top:1px;">{i}</span>'
                     f'<span style="font-size:13px;color:#333;line-height:1.6;font-weight:500;">{m}</span></div>')
        return rows

    def week_row_html(offset):
        d   = today + timedelta(days=offset)
        ds  = d.strftime("%Y%m%d")
        dow_list = ["월","화","수","목","금","토","일"]
        dow = dow_list[d.weekday()]
        if offset == 0:          day_color = "#185FA5"
        elif d.weekday() == 5:   day_color = "#185FA5"
        elif d.weekday() == 6:   day_color = "#A32D2D"
        else:                    day_color = "#222"
        tag = (" <span style='font-size:10px;color:#aaa;'>(오늘)</span>" if offset == 0
               else " <span style='font-size:10px;color:#aaa;'>(내일)</span>" if offset == 1
               else "")
        if ds not in daily:
            return (f'<div style="display:flex;align-items:center;padding:6px 0;border-bottom:0.5px solid #eee;">'
                    f'<span style="font-size:13px;font-weight:500;width:70px;color:{day_color};">{dow}요일{tag}</span>'
                    f'<span style="flex:1;font-size:12px;color:#888;">─</span>'
                    f'<span style="font-size:12px;color:#378ADD;width:32px;text-align:center;">─</span>'
                    f'<span style="font-size:12px;min-width:70px;text-align:right;">─</span></div>')
        dd = daily[ds]
        sky_rep  = sky_label(Counter(dd["SKY"]).most_common(1)[0][0]) if dd["SKY"] else "─"
        pty_rep  = pty_label(Counter(dd["PTY"]).most_common(1)[0][0]) if dd["PTY"] else ""
        pop_rep  = f"{max(dd['POP'])}%" if dd["POP"] else "─"
        tmx_rep  = f"{dd['TMX']}°" if dd["TMX"] else "─"
        tmn_rep  = f"{dd['TMN']}°" if dd["TMN"] else "─"
        sky_disp = (sky_rep + (" / " + pty_rep if pty_rep and pty_rep != "없음" else "")).strip()
        return (f'<div style="display:flex;align-items:center;padding:6px 0;border-bottom:0.5px solid #eee;">'
                f'<span style="font-size:13px;font-weight:500;width:70px;color:{day_color};">{dow}요일{tag}</span>'
                f'<span style="flex:1;font-size:12px;color:#555;">{sky_disp}</span>'
                f'<span style="font-size:12px;color:#378ADD;width:32px;text-align:center;">{pop_rep}</span>'
                f'<span style="font-size:12px;min-width:70px;text-align:right;">'
                f'<span style="color:#D85A30;font-weight:500;">{tmx_rep}</span>'
                f'<span style="color:#378ADD;margin-left:4px;">{tmn_rep}</span></span></div>')

    # ── 기온 바 ─────────────────────────────────────────────────────
    slots      = ["0000","0300","0600","0900","1200","1500","1800","2100"]
    slot_labels = {"0000":"00시","0300":"03시","0600":"06시","0900":"09시",
                   "1200":"12시","1500":"15시","1800":"18시","2100":"21시"}
    bars_html = "".join(temp_bar_html(slot_labels[sl], hour_temps[sl])
                        for sl in slots if sl in hour_temps)
    if not bars_html:
        bars_html = '<span style="font-size:12px;color:#aaa;">데이터 없음</span>'

    # ── 주간 ────────────────────────────────────────────────────────
    week_html = "".join(week_row_html(i) for i in range(7))

    # ── 특보 ────────────────────────────────────────────────────────
    if warn_items is None:
        warn_html = '<div style="font-size:13px;color:#888;">특보 정보를 가져오지 못했습니다.</div>'
    elif len(warn_items) == 0:
        warn_html = ('<div style="display:flex;align-items:center;gap:10px;">'
                     '<div style="width:28px;height:28px;border-radius:50%;background:#E1F5EE;'
                     'display:flex;align-items:center;justify-content:center;flex-shrink:0;">'
                     '<svg width="14" height="14" viewBox="0 0 14 14" fill="none">'
                     '<circle cx="7" cy="7" r="5.5" stroke="#0F6E56" stroke-width="1.2"/>'
                     '<path d="M5 7l1.5 1.5L9 5" stroke="#0F6E56" stroke-width="1.2" '
                     'stroke-linecap="round" stroke-linejoin="round"/></svg></div>'
                     f'<span style="font-size:13px;color:#555;">현재 {region_name} 지역 발효 중인 기상 특보 없음</span></div>')
    else:
        warn_html = ""
        for w in warn_items:
            warn_html += (f'<div style="padding:8px 0;border-bottom:0.5px solid #eee;">'
                          f'<div style="font-size:13px;font-weight:500;color:#A32D2D;margin-bottom:4px;">'
                          f'⚠ {w["title"]}</div>'
                          f'<div style="font-size:11px;color:#888;margin-bottom:4px;">발표: {w["date"]}</div>'
                          f'<div style="font-size:12px;color:#555;line-height:1.5;">{w["desc"][:150]}...</div></div>')

    # ── 최종 HTML 조립 ──────────────────────────────────────────────
    html = f"""
    <div style="max-width:390px;background:#f0f2f5;padding:12px;
                border-radius:16px;font-family:sans-serif;">

      <!-- 헤더 -->
      <div style="background:linear-gradient(135deg,#1a3a5c,#2563a8);border-radius:12px;
                  padding:16px;color:#fff;margin-bottom:10px;">
        <div style="display:flex;justify-content:space-between;align-items:flex-start;">
          <div>
            <div style="font-size:13px;opacity:.75;margin-bottom:2px;">기상현황 브리핑</div>
            <div style="font-size:20px;font-weight:600;">{region_name}</div>
          </div>
          <div style="text-align:right;font-size:11px;opacity:.65;line-height:1.8;">
            {today.strftime('%Y.%m.%d')}<br>{today.strftime('%H:%M')} 현재
          </div>
        </div>
      </div>

      {card('[1] 오늘의 날씨',
        row("하늘 상태", sky_label(today_sky)) +
        row("강수 형태", pty_label(today_pty)) +
        row("강수 확률", f"{today_pop}%") +
        row("최고 / 최저",
            f'<span style="color:#D85A30;">{today_tmx}°C</span>'
            f' / <span style="color:#185FA5;">{today_tmn}°C</span>') +
        row("현재 습도", f"{cur_hum}%") +
        row("현재 풍속", f"{cur_wind} m/s &nbsp;{wind_dir_label(cur_wdir)}")
      )}

      {card('[2] 내일 기상 예측',
        row("하늘 상태", sky_label(tmr_sky)) +
        row("강수 형태", pty_label(tmr_pty)) +
        row("강수 확률", f"{tmr_pop}%") +
        row("최고 / 최저",
            f'<span style="color:#D85A30;">{tmr_tmx}°C</span>'
            f' / <span style="color:#185FA5;">{tmr_tmn}°C</span>')
      )}

      {card('[3] 현재 기온',
        stat2("현재 기온", f"{cur_temp}°C", "실황",
              "체감 기온", chill, "풍속 보정") +
        row("일최고 예상", f"{today_tmx}°C", "color:#D85A30;") +
        row("일최저 예상", f"{today_tmn}°C", "color:#185FA5;")
      )}

      {card('[4] 기온 변화 (오늘)', bars_html)}

      {card('[5] 가시거리',
        row("추정 가시거리", vis_str) +
        f'<div style="margin-top:6px;font-size:11px;color:#aaa;">'
        f'하늘: {sky_label(today_sky)} / 강수: {pty_label(today_pty)} / 풍속: {cur_wind} m/s</div>'
      )}

      {card('[6] 주간 요일별 예상 날씨',
        '<div style="display:flex;margin-bottom:6px;padding-bottom:4px;border-bottom:0.5px solid #eee;">'
        '<span style="font-size:10px;color:#aaa;width:70px;">요일</span>'
        '<span style="font-size:10px;color:#aaa;flex:1;">날씨</span>'
        '<span style="font-size:10px;color:#aaa;width:32px;text-align:center;">강수</span>'
        '<span style="font-size:10px;color:#aaa;min-width:70px;text-align:right;">최고/최저</span>'
        '</div>' + week_html
      )}

      {card('[7] 기상 특보', warn_html)}

      {card('[8] 화재 위험',
        row("위험 수준", fire_badge(f_level)) +
        row("판정 근거",
            f"기온 {cur_temp}°C / 습도 {cur_hum}% / 풍속 {cur_wind} m/s",
            "font-size:11px;") +
        row("조치 사항", f_action, "font-size:11px;max-width:55%;")
      )}

      {card('[9] 사고 예방 조치사항', measure_items_html(m_list), danger=True)}

      <div style="text-align:center;font-size:10px;color:#aaa;padding-bottom:8px;line-height:1.8;">
        ※ 기상청 공공데이터 기반 자동 생성<br>
        ※ 출처: 기상청 단기예보 서비스 (data.go.kr)<br>
        ※ 작성: {today.strftime('%Y-%m-%d %H:%M')} 자동 생성
      </div>
    </div>
    """
    display(HTML(html))


#5. UI 구성 및 실행 ==========

sido_list = list(REGION_DATA.keys())

# ── 지역 드롭다운 ────────────────────────────────────────────────
sido_dd = widgets.Dropdown(
    options=sido_list,
    value="대전광역시",
    description="지역:",
    style={"description_width": "40px"},
    layout=widgets.Layout(width="210px")
)

# ── 출력 영역 ────────────────────────────────────────────────────
out = widgets.Output()

# ── 숨겨진 실제 ipywidgets 버튼 (Python 콜백 수신 전용) ──────────
_hidden_btn = widgets.Button(
    description="",
    layout=widgets.Layout(width="1px", height="1px",
                          visibility="hidden", margin="0", padding="0")
)

# ── 화면에 표시되는 현대적 HTML 버튼 ───────────────────────────
btn_html = widgets.HTML("""
<style>
  .wx-btn {
    display: inline-flex;
    align-items: center;
    gap: 7px;
    padding: 10px 24px;
    background: linear-gradient(135deg, #1a3a5c 0%, #2563a8 100%);
    color: #fff;
    font-family: sans-serif;
    font-size: 14px;
    font-weight: 600;
    letter-spacing: 0.4px;
    border: none;
    border-radius: 50px;
    cursor: pointer;
    box-shadow: 0 4px 14px rgba(37,99,168,0.40);
    transition: transform 0.15s ease, box-shadow 0.15s ease;
    user-select: none;
    outline: none;
  }
  .wx-btn:hover {
    box-shadow: 0 6px 20px rgba(37,99,168,0.50);
    transform: translateY(-2px);
  }
  .wx-btn:active {
    transform: translateY(0);
    box-shadow: 0 2px 8px rgba(37,99,168,0.30);
  }
  .wx-btn svg { flex-shrink: 0; }
</style>

<button class="wx-btn" onclick="(function(){
  var btns = document.querySelectorAll(
    'button.widget-button, .widget-button button'
  );
  for(var i=0; i<btns.length; i++){
    var s = btns[i].style;
    if(s.visibility==='hidden' || s.width==='1px'){
      btns[i].click(); break;
    }
  }
})()">
  <svg width="15" height="15" viewBox="0 0 15 15" fill="none">
    <circle cx="6" cy="6" r="4.2" stroke="white" stroke-width="1.6"/>
    <path d="M9.2 9.2L13 13" stroke="white" stroke-width="1.6" stroke-linecap="round"/>
  </svg>
  날씨 조회
</button>
""")

# ── 버튼 클릭 핸들러 ─────────────────────────────────────────────
def on_search_click(b):
    """
    ✅ 핵심: sido_dd.value를 클릭 시점에 새로 읽어서
             선택된 지역의 최신 날씨를 항상 새로 호출합니다.
    """
    sido    = sido_dd.value                          # 현재 선택된 시/도
    sigungu = list(REGION_DATA[sido].keys())[0]      # 대표 도시
    nx, ny, warn_kw = REGION_DATA[sido][sigungu]

    # 지역명 포맷
    if sido.endswith(("특별시", "광역시", "특별자치시", "특별자치도")):
        region_name = sido
    else:
        region_name = f"{sido} ({sigungu})"

    # 로딩 메시지 → 결과로 교체
    with out:
        clear_output(wait=True)
        display(HTML(
            f'<div style="font-family:sans-serif;padding:16px 0;">'
            f'<div style="display:inline-flex;align-items:center;gap:10px;">'
            f'<div style="width:18px;height:18px;border:2px solid #2563a8;'
            f'border-top-color:transparent;border-radius:50%;'
            f'animation:spin .8s linear infinite;"></div>'
            f'<style>@keyframes spin{{to{{transform:rotate(360deg)}}}}</style>'
            f'<span style="font-size:13px;color:#555;">⏳ {region_name} 날씨 정보를 불러오는 중…</span>'
            f'</div></div>'
        ))
        fetch_and_render(region_name, nx, ny, warn_kw)

_hidden_btn.on_click(on_search_click)

# ── 헤더 타이틀 ──────────────────────────────────────────────────
title_html = HTML(
    '<div style="font-family:sans-serif;'
    'background:linear-gradient(135deg,#1a3a5c 0%,#2563a8 100%);'
    'color:#fff;border-radius:12px;padding:14px 18px;margin-bottom:12px;max-width:430px;">'
    '<div style="font-size:19px;font-weight:700;margin-bottom:3px;">🌤️ 지역별 기상 브리핑</div>'
    '<div style="font-size:12px;opacity:.75;">지역을 선택하고 날씨 조회 버튼을 누르세요</div>'
    '</div>'
)

display(title_html)
display(widgets.HBox(
    [sido_dd, btn_html, _hidden_btn],
    layout=widgets.Layout(gap="10px", align_items="center", margin="0 0 14px 0")
))
display(out)

# ── 초기 자동 실행 (대전) ────────────────────────────────────────
on_search_click(None)

✅ 설정 완료
✅ 함수 정의 완료


Output()